# Fluorescence spot and fusion-event analysis

Single-channel hemifusion and dual-channel two-step fusion

This notebook detects fluorescent spots in the **DV (left) channel**, extracts background-corrected intensity traces, and identifies positive transitions using complementary derivative and intensity-level detectors. For dual-channel recordings, the LEM (right) channel is analyzed **at positions matched to the DV-channel spots**, without independent spot detection.

## Workflow
1. Load a TIFF image sequence; optionally split side-by-side channels.
2. Estimate static background maps from an averaged reference image and detect local maxima in the primary channel.
3. Extract spot-associated fluorescence traces (default: pixel with strongest positive derivative).
4. Detect and validate fluorescence increases with the same event detection functions for either acquisition mode.
5. In dual-channel mode, match events within a time tolerance and report per-pair frame differences. 
6. Export per-spot traces, per-event tables and diagnostic plots.


## 1. Imports

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import tifffile as tiff
import matplotlib.pyplot as plt
from scipy.ndimage import maximum_filter
from scipy import interpolate
from skimage.restoration import rolling_ball
from skimage.draw import disk
from tqdm.auto import tqdm


## 2. Configuration

Specify `INPUT_TIFF`, `OUTPUT_DIR`, `ACQUISITION_MODE` and actual frame time. Right-channel settings are used only for dual-channel recordings. 

In [ ]:
# User paths and acquisition settings 
INPUT_TIFF = "path/to/recording.tif"
OUTPUT_DIR = "analysis_output"
ACQUISITION_MODE = "dual"  # "single" or "dual" 
FRAME_INTERVAL_S = None    # Unit: seconds 

# Spot detection / fluorescence extraction
CHECK_FIRST_N_FRAMES = 3000
SPOT_RADIUS_PX = 3
NOISE_TOLERANCE = 10.0
ROLLING_BALL_RADIUS = 100
CORNER_K = 1

# ROI extraction mode: "mean_disk" | "p90_disk" | "max_disk" | "best_pixel_by_derivative"
ROI_MODE = "best_pixel_by_derivative"

# For best-pixel mode (only used if ROI_MODE == "best_pixel_by_derivative")
BESTPIX_MIN_PEAK_INTENSITY = None  # e.g. 20.0
# Optional scoring window in ORIGINAL frame indices; set None to scan all
BESTPIX_SCORE_WINDOW = None        # e.g. (500, 4000)
# If True, choose the best pixel independently in the right-channel ROI instead of
# forcing the right trace to use the left-channel picked pixel coordinate.
RIGHT_BESTPIX_INDEPENDENT = True
# When the left channel has detected events, right-channel best-pixel selection is
# scored preferentially within +/- this many frames of those left events.
RIGHT_BESTPIX_EVENT_HALF_WINDOW = 350
# Search the right-channel best pixel around the left picked pixel.
RIGHT_SEARCH_RADIUS_PX = SPOT_RADIUS_PX
RIGHT_SEARCH_CENTER = "picked_left"  # "picked_left" | "spot_center"

# Event detector params 
EVENT_PARAMS = dict(
    t_start=0,
    smooth_N=100,
    spline_k=2,
    dev_smooth_N=100,
    shift=100,
    binsize=100,
    pos_abs_thr=0.002,
    neg_abs_thr=0.005,
    std_factor=4.0,
    ignore_half_window=100,
    debug_plot=False,
)

# Supplemental level-shift detector: rescues slower/smaller positive steps that
# do not create a large enough derivative peak.
LEVEL_SHIFT_PARAMS = dict(
    enabled=True,
    t_start=0,
    smooth_N=25,
    window=80,
    gap=10,
    step=5,
    min_abs_delta=0.03,
    robust_z=2.5,
    refractory=150,
    merge_tol=25,
    debug_plot=False,
)

# Final post-hoc event validation. This removes derivative false positives caused
# by slow drift/noise. 
LEFT_EVENT_FILTER_PARAMS = dict(
    enabled=True,
    t_start=0,
    smooth_N=25,
    pre_window=90,
    post_window=120,
    gap=10,
    min_abs_delta=0.08,
    robust_z=3,
    refractory=300,
    debug_plot=False,
)

RIGHT_EVENT_FILTER_PARAMS = dict(
    enabled=True,
    t_start=0,
    smooth_N=25,
    pre_window=90,
    post_window=120,
    gap=10,
    min_abs_delta=0.12,
    robust_z=3.5,
    refractory=450,
    debug_plot=False,
)

# Match events by nearest absolute frame difference.
# Matching tolerance (frames)
MATCH_TOL = 1500

# Diagnostics
SHOW_DETECTION_OVERLAY = True
PLOT_FIRST_N_MATCHED_SPOTS = 3   # set 0 to skip


## 3. Loading, background correction and primary-channel spot detection

In [ ]:
def read_tiff_stack_gray(path: str) -> np.ndarray:
    """Read multi-page TIFF into (T,H,W) float32 grayscale array."""
    with tiff.TiffFile(path) as tif:
        frames = []
        for p in tif.pages:
            arr = p.asarray()
            if arr.ndim == 3:  # RGB(A) -> grayscale
                arr = arr[..., :3].mean(axis=-1)
            frames.append(arr.astype(np.float32))
    return np.stack(frames, axis=0)

def split_left_right(vol: np.ndarray):
    """Split stitched frames into left/right channel stacks: (T,H,W/2) each."""
    if vol.ndim != 3:
        raise ValueError("Expected volume shape (T,H,W).")
    T, H, W = vol.shape
    if (W % 2) != 0:
        raise ValueError("Frame width must be even to split into two equal channels.")
    mid = W // 2
    return vol[:, :, :mid], vol[:, :, mid:]

def average_first_n(vol: np.ndarray, n: int) -> np.ndarray:
    n = int(max(1, min(n, vol.shape[0])))
    return vol[:n].mean(axis=0)

def estimate_background(avg_img: np.ndarray, rolling_ball_radius: int = 100) -> np.ndarray:
    bg = rolling_ball(avg_img, radius=int(rolling_ball_radius))
    return bg.astype(np.float32)


def _corner_mean(win: np.ndarray, k: int = 1) -> float:
    """Mean of the 4 corners of a window (optionally averaged over (2k+1)x(2k+1) blocks at corners)."""
    H, W = win.shape
    if H != W or (H % 2) != 1:
        raise ValueError("Window must be odd-sized square.")
    def block_mean(y, x):
        y0, y1 = max(0, y - k), min(H, y + k + 1)
        x0, x1 = max(0, x - k), min(W, x + k + 1)
        return float(win[y0:y1, x0:x1].mean())
    m1 = block_mean(0, 0)
    m2 = block_mean(0, W - 1)
    m3 = block_mean(H - 1, 0)
    m4 = block_mean(H - 1, W - 1)
    return (m1 + m2 + m3 + m4) / 4.0

def detect_spots_local_max(img: np.ndarray, spot_radius: int, noise_tolerance: float, corner_k: int = 1):
    """Detect spot centers (y,x) using local maxima + corner mean + noise tolerance."""
    if img.ndim != 2:
        raise ValueError("detect_spots_local_max expects a 2D grayscale image.")
    r = int(spot_radius)
    H, W = img.shape
    size = 2 * r + 1

    max_f = maximum_filter(img, size=size, mode="nearest")
    is_local_max = (img == max_f)
    ys, xs = np.nonzero(is_local_max)

    centers = []
    for y, x in zip(ys, xs):
        if y < r or y >= H - r or x < r or x >= W - r:
            continue
        win = img[y - r:y + r + 1, x - r:x + r + 1]
        peak = float(img[y, x])
        cm = _corner_mean(win, k=corner_k)
        if peak >= (cm + noise_tolerance):
            centers.append((int(y), int(x)))
    return centers

def show_detection_compare(left_first: np.ndarray, centers: list, spot_radius: int):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    axes[0].imshow(left_first, cmap="gray")
    axes[0].set_title("Left channel - avg/check image")
    axes[0].axis("off")

    axes[1].imshow(left_first, cmap="gray")
    for (y, x) in centers:
        axes[1].add_patch(plt.Circle((x, y), radius=spot_radius, fill=False, edgecolor='white', linewidth=0.8))
    axes[1].set_title("Detected spots (overlay)")
    axes[1].axis("off")
    plt.show()


## 4. Fluorescence trace extraction

The default ROI mode selects the pixel with the largest positive smoothed-derivative response inside a circular search region. Alternative ROI summary modes are provided for exploratory comparisons.

In [ ]:
# ---------- Trace extraction (4 ROI modes) ----------

def _disk_coords(H, W, y, x, radius):
    rr, cc = disk((y, x), int(radius), shape=(H, W))
    return rr.astype(int), cc.astype(int)

def extract_trace_for_roi(vol: np.ndarray, bg: np.ndarray, y: int, x: int, radius: int, mode: str):
    """Return (T,) trace for a single channel stack."""
    T, H, W = vol.shape
    rr, cc = _disk_coords(H, W, y, x, radius)
    # background subtract frame-by-frame using a static bg map
    if mode == "mean_disk":
        return np.array([(vol[t] - bg)[rr, cc].mean() for t in range(T)], dtype=float)
    if mode == "p90_disk":
        return np.array([np.percentile((vol[t] - bg)[rr, cc], 90) for t in range(T)], dtype=float)
    if mode == "max_disk":
        return np.array([(vol[t] - bg)[rr, cc].max() for t in range(T)], dtype=float)
    raise ValueError(f"Unknown mode: {mode}")

def extract_pixel_traces(vol: np.ndarray, bg: np.ndarray, y: int, x: int, radius: int):
    """Return traces for every pixel within disk: traces shape (Npix, T), plus pixel coords arrays (ys,xs)."""
    T, H, W = vol.shape
    rr, cc = _disk_coords(H, W, y, x, radius)
    N = len(rr)
    traces = np.empty((N, T), dtype=float)
    for t in range(T):
        traces[:, t] = (vol[t] - bg)[rr, cc]
    return traces, rr, cc


## 5. Candidate event detection and validation

The derivative detector targets rapid positive intensity transitions. The level-shift detector identifies sustained increases that can be missed by derivative-based detection. Events are merged and validated against pre/post intensity differences and robust noise estimates. 

In [ ]:
# Derivative-based detection of positive fluorescence transitions

def moving_average(y: np.ndarray, N: int) -> np.ndarray:
    if N <= 1:
        return y.astype(float)
    kernel = np.ones(N, dtype=float) / float(N)
    return np.convolve(y, kernel, mode="valid")

def _dev_signal(y: np.ndarray, t_start: int = 0, smooth_N: int = 100, spline_k: int = 2, dev_smooth_N: int = 100):
    """Return (dev_s, x_dev_int) used by get_events_like; helpful for scoring pixels."""
    y = np.asarray(y, dtype=float)
    x = np.arange(len(y), dtype=float)

    y_s = moving_average(y, smooth_N)
    x_s = moving_average(x, smooth_N)

    ymax = np.max(y_s)
    if ymax <= 0:
        return None, None
    y_s = y_s / ymax

    tck = interpolate.splrep(x_s, y_s, k=spline_k, s=0)
    dev_1 = interpolate.splev(x_s, tck, der=1)

    dev_s = moving_average(dev_1, dev_smooth_N)
    x_dev = x_s[(dev_smooth_N - 1):]
    x_dev_int = np.rint(x_dev).astype(int) + int(t_start)
    return dev_s, x_dev_int

def get_events_like(
    y: np.ndarray,
    t_start: int = 0,
    smooth_N: int = 100,
    spline_k: int = 2,
    dev_smooth_N: int = 100,
    shift: int = 100,
    binsize: int = 100,
    pos_abs_thr: float = 0.002,
    neg_abs_thr: float = 0.005,
    std_factor: float = 5.0,
    ignore_half_window: int = 100,
    debug_plot: bool = False,
):
    y = np.asarray(y, dtype=float)
    if y.size < (smooth_N + dev_smooth_N + 10):
        return []

    dev_s, x_dev_int = _dev_signal(y, t_start=t_start, smooth_N=smooth_N, spline_k=spline_k, dev_smooth_N=dev_smooth_N)
    if dev_s is None:
        return []

    if debug_plot:
        plt.figure(figsize=(6,3))
        plt.plot(dev_s)
        plt.title("dev_s (smoothed derivative)")
        plt.xlabel("index"); plt.ylabel("dev_s")
        plt.show()

    steps, ignores = [], []
    L = len(dev_s)

    for end in range(binsize, L + 1, binsize):
        w0, w1 = end - binsize, end

        a_idx = np.r_[max(0, w0 - shift - binsize):max(0, w0 - shift),
                      min(L, w1 + shift):min(L, w1 + shift + binsize)]
        a_idx_before = np.r_[max(0, w0 - shift - binsize):max(0, w0 - shift)]
        a_idx_after  = np.r_[min(L, w1 + shift):min(L, w1 + shift + binsize)]

        a_idx = np.unique(a_idx)
        if a_idx.size < 10:
            continue

        bg = dev_s[a_idx]
        # use smoothed normalized intensity trend for direction check
        y_s = moving_average(y, smooth_N)
        y_s = y_s / (np.max(y_s) if np.max(y_s) > 0 else 1.0)
        bg_before = y_s[a_idx_before] if a_idx_before.size else y_s[:1]
        bg_after  = y_s[a_idx_after] if a_idx_after.size else y_s[-1:]

        local = dev_s[w0:w1]
        cond_pos = (local > (std_factor * np.std(bg))) & (local > pos_abs_thr) & (np.mean(bg_before) < np.mean(bg_after))
        pos_hits = np.where(cond_pos)[0]
        if pos_hits.size > 0:
            steps.append(w0 + int(pos_hits[0]))

        local2 = dev_s[w0:min(L, w1 + binsize)]
        cond_neg = (local2 < -neg_abs_thr) | (local2 < -(std_factor * np.std(bg)))
        neg_hits = np.where(cond_neg)[0]
        if neg_hits.size > 0:
            ignores.append(w0 + int(neg_hits[0]))

    steps = sorted(set(steps))
    ignores = sorted(set(ignores))

    ignore_set = set()
    for ig in ignores:
        for j in range(ig - ignore_half_window, ig + ignore_half_window + 1):
            ignore_set.add(j)

    steps_clean = [s for s in steps if s not in ignore_set]

    events = []
    for s in steps_clean:
        if 0 <= s < len(x_dev_int):
            events.append(int(x_dev_int[s]))
    return sorted(set(events))

# Complementary level-shift detection
# The derivative detector above is ok for sharp steps, but it can miss slow
# plateau changes. This fallback explicitly compares pre/post windows on a lightly
# smoothed, normalized trace.

def _robust_sigma(vals: np.ndarray) -> float:
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return 0.0
    med = np.median(vals)
    mad = np.median(np.abs(vals - med))
    if mad > 0:
        return float(1.4826 * mad)
    return float(np.std(vals))

def _merge_close_events(events, tol: int = 25):
    """Merge nearby event times; keep the earliest time in each cluster."""
    events = sorted(int(e) for e in events)
    if not events:
        return []
    merged = [events[0]]
    for e in events[1:]:
        if e - merged[-1] <= int(tol):
            continue
        merged.append(e)
    return merged

def detect_level_shift_events(
    y: np.ndarray,
    t_start: int = 0,
    smooth_N: int = 25,
    window: int = 80,
    gap: int = 10,
    step: int = 5,
    min_abs_delta: float = 0.03,
    robust_z: float = 2.5,
    refractory: int = 150,
    merge_tol: int = 25,
    debug_plot: bool = False,
):
    """
    Detect positive step-like level shifts by comparing the median intensity before
    and after each candidate frame. Values are normalized by the smoothed trace max.
    """
    y = np.asarray(y, dtype=float)
    if y.size < (2 * window + 2 * gap + smooth_N + 5):
        return []

    y_s = moving_average(y, smooth_N)
    x_s = moving_average(np.arange(len(y), dtype=float), smooth_N)
    ymax = np.max(y_s)
    if ymax <= 0:
        return []
    y_n = y_s / ymax

    candidates = []
    start = int(window + gap)
    stop = int(len(y_n) - window - gap)
    for c in range(start, stop, int(step)):
        before = y_n[c - gap - window:c - gap]
        after = y_n[c + gap:c + gap + window]
        if before.size < 10 or after.size < 10:
            continue
        delta = float(np.median(after) - np.median(before))
        # Estimate noise within each plateau separately. If before/after are mixed
        # together, the step itself inflates sigma and weakens the detector.
        sigma_before = _robust_sigma(before - np.median(before))
        sigma_after = _robust_sigma(after - np.median(after))
        sigma = max(sigma_before, sigma_after, 1e-12)
        score = delta / sigma
        if (delta >= min_abs_delta) and (score >= robust_z):
            frame = int(round(x_s[c])) + int(t_start)
            candidates.append((score, delta, frame))

    # Non-maximum suppression by score so one broad step does not generate many events.
    kept = []
    for score, delta, frame in sorted(candidates, reverse=True):
        if all(abs(frame - k) > int(refractory) for k in kept):
            kept.append(frame)

    events = _merge_close_events(sorted(kept), tol=merge_tol)

    if debug_plot:
        plt.figure(figsize=(7, 3))
        plt.plot(x_s + int(t_start), y_n, label="smoothed normalized trace")
        for ev in events:
            plt.axvline(ev, color="green", linestyle="--", alpha=0.6)
        plt.title("level-shift fallback events")
        plt.xlabel("Frame")
        plt.ylabel("Normalized intensity")
        plt.tight_layout()
        plt.show()

    return events

def get_events_combined(y: np.ndarray, derivative_params: dict, level_shift_params=None):
    """Union derivative events with optional level-shift fallback events."""
    derivative_events = get_events_like(y, **derivative_params)

    if not level_shift_params or not level_shift_params.get("enabled", False):
        return derivative_events

    params = dict(level_shift_params)
    params.pop("enabled", None)
    level_events = detect_level_shift_events(y, **params)
    merge_tol = int(level_shift_params.get("merge_tol", 25))
    return _merge_close_events(derivative_events + level_events, tol=merge_tol)


# Post-detection validation
# This filter removes calls that do not look like sustained positive level changes.

def filter_step_events(
    y: np.ndarray,
    events,
    t_start: int = 0,
    smooth_N: int = 25,
    pre_window: int = 90,
    post_window: int = 120,
    gap: int = 10,
    min_abs_delta: float = 0.08,
    robust_z: float = 3.0,
    refractory: int = 350,
    debug_plot: bool = False,
):
    """
    Keep only events with a sustained positive pre/post median shift.
    Thresholds operate on a 0-1 normalized smoothed trace for each ROI.
    """
    events = sorted(int(e) for e in events)
    if not events:
        return []

    y = np.asarray(y, dtype=float)
    if y.size < (pre_window + post_window + 2 * gap + smooth_N + 5):
        return events

    y_s = moving_average(y, smooth_N)
    x_s = moving_average(np.arange(len(y), dtype=float), smooth_N) + int(t_start)
    finite = np.isfinite(y_s)
    if not np.any(finite):
        return []
    ymax = np.nanmax(y_s)
    ymin = np.nanmin(y_s)
    if not np.isfinite(ymax) or ymax <= ymin:
        return []
    y_n = (y_s - ymin) / (ymax - ymin)

    candidates = []
    for ev in events:
        c = int(np.argmin(np.abs(x_s - ev)))
        b0 = c - gap - pre_window
        b1 = c - gap
        a0 = c + gap
        a1 = c + gap + post_window
        if b0 < 0 or a1 > len(y_n):
            continue
        before = y_n[b0:b1]
        after = y_n[a0:a1]
        if before.size < 10 or after.size < 10:
            continue
        delta = float(np.median(after) - np.median(before))
        sigma_before = _robust_sigma(before - np.median(before))
        sigma_after = _robust_sigma(after - np.median(after))
        sigma = max(sigma_before, sigma_after, 1e-12)
        score = delta / sigma
        if (delta >= min_abs_delta) and (score >= robust_z):
            candidates.append((score, delta, ev))

    kept = []
    for score, delta, ev in sorted(candidates, reverse=True):
        if all(abs(ev - old_ev) > int(refractory) for old_ev in kept):
            kept.append(ev)
    kept = sorted(kept)

    if debug_plot:
        plt.figure(figsize=(7, 3))
        plt.plot(x_s, y_n, label="smoothed normalized trace")
        for ev in events:
            plt.axvline(ev, color="gray", linestyle=":", alpha=0.35)
        for ev in kept:
            plt.axvline(ev, color="green", linestyle="--", alpha=0.7)
        plt.title("post-hoc event validation")
        plt.xlabel("Frame")
        plt.ylabel("Normalized intensity")
        plt.tight_layout()
        plt.show()

    return kept


## 6. Representative-pixel selection and cross-channel event matching

In [ ]:
def choose_best_pixel_by_derivative(
    pixel_traces,
    ys,
    xs,
    score_window=None,
    dev_params=None,
    min_peak_intensity=None
    ):
    """
    Pick the pixel inside the ROI whose smoothed-derivative peak is largest.

    score_window: (t0, t1) in ORIGINAL frame index. If None, score across whole trace.
    dev_params: dict passed to _dev_signal (smooth_N, spline_k, dev_smooth_N, t_start)
    min_peak_intensity: optional absolute intensity gate (after bg-sub).
    """
    if dev_params is None:
        dev_params = dict(t_start=0, smooth_N=100, spline_k=2, dev_smooth_N=100)

    T = pixel_traces.shape[1]
    best_i = None
    best_score = -np.inf

    # quick intensity gate based on max intensity
    max_int = pixel_traces.max(axis=1)

    for i in range(pixel_traces.shape[0]):
        if (min_peak_intensity is not None) and (max_int[i] < min_peak_intensity):
            continue
        dev_s, x_dev_int = _dev_signal(pixel_traces[i], **dev_params)
        if dev_s is None:
            continue
        # map desired window to dev_s indices via x_dev_int
        if score_window is not None:
            t0, t1 = score_window
            mask = (x_dev_int >= t0) & (x_dev_int <= t1)
            if not np.any(mask):
                continue
            score = float(np.max(dev_s[mask]))
        else:
            score = float(np.max(dev_s))
        if score > best_score:
            best_score = score
            best_i = i

    if best_i is None:
        # fallback: choose brightest pixel by max intensity
        best_i = int(np.argmax(max_int))
        best_score = float('nan')

    return best_i, best_score, int(ys[best_i]), int(xs[best_i])


def choose_best_pixel_near_events(
    pixel_traces,
    ys,
    xs,
    anchor_events,
    half_window: int = 300,
    dev_params=None,
    min_peak_intensity=None,
):
    """
    Pick the pixel whose positive derivative is strongest near anchor event times.
    The liposome signal may be offset within the ROI.
    """
    anchor_events = [int(e) for e in anchor_events if pd.notna(e)]
    if not anchor_events:
        return choose_best_pixel_by_derivative(
            pixel_traces, ys, xs,
            score_window=None,
            dev_params=dev_params,
            min_peak_intensity=min_peak_intensity,
        )

    if dev_params is None:
        dev_params = dict(t_start=0, smooth_N=100, spline_k=2, dev_smooth_N=100)

    best_i = None
    best_score = -np.inf
    best_event_distance = np.inf
    max_int = pixel_traces.max(axis=1)

    for i in range(pixel_traces.shape[0]):
        if (min_peak_intensity is not None) and (max_int[i] < min_peak_intensity):
            continue
        dev_s, x_dev_int = _dev_signal(pixel_traces[i], **dev_params)
        if dev_s is None:
            continue

        masks = [np.abs(x_dev_int - ev) <= int(half_window) for ev in anchor_events]
        mask = np.logical_or.reduce(masks) if masks else np.zeros_like(x_dev_int, dtype=bool)
        if not np.any(mask):
            continue

        idxs = np.where(mask)[0]
        local_i = idxs[int(np.argmax(dev_s[mask]))]
        score = float(dev_s[local_i])
        event_distance = min(abs(int(x_dev_int[local_i]) - ev) for ev in anchor_events)

        if (score > best_score) or (score == best_score and event_distance < best_event_distance):
            best_score = score
            best_i = i
            best_event_distance = event_distance

    if best_i is None:
        return choose_best_pixel_by_derivative(
            pixel_traces, ys, xs,
            score_window=None,
            dev_params=dev_params,
            min_peak_intensity=min_peak_intensity,
        )

    return best_i, best_score, int(ys[best_i]), int(xs[best_i])


def match_events(left_events, right_events, tol: int = 3):
    """Match left/right events by globally closest frame differences."""
    left_events = sorted(int(e) for e in left_events)
    right_events = sorted(int(e) for e in right_events)
    candidates = sorted(
        (abs(le - re), le, re)
        for le in left_events
        for re in right_events
        if abs(le - re) <= int(tol)
    )
    matched = []
    used_l = set()
    used_r = set()
    for _, le, re in candidates:
        if le in used_l or re in used_r:
            continue
        matched.append((le, re))
        used_l.add(le)
        used_r.add(re)
    return sorted(matched)


def normalize_trace_for_plot(y: np.ndarray, robust: bool = True):
    """Normalize one trace to 0-1 for plotting only; raw values in df stay unchanged."""
    y = np.asarray(y, dtype=float)
    if y.size == 0:
        return y
    if robust:
        lo, hi = np.percentile(y[np.isfinite(y)], [1, 99])
    else:
        lo, hi = np.nanmin(y), np.nanmax(y)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(y, dtype=float)
    return np.clip((y - lo) / (hi - lo), 0, 1)

def show_matched_event_overlay(avg_left: np.ndarray, df: pd.DataFrame, spot_radius: int, max_points=None):
    """
    Overlay only spots with >=1 matched event on the left-channel average/check image.
    """
    matched_df = df[df["n_matched_events"] > 0].copy()
    if max_points is not None:
        matched_df = matched_df.head(int(max_points))

    plt.figure(figsize=(6, 6))
    plt.imshow(avg_left, cmap="gray")
    for _, r in matched_df.iterrows():
        x = int(r["picked_left_x"]) if "picked_left_x" in matched_df.columns else int(r["picked_x"]) if "picked_x" in matched_df.columns else int(r["x"])
        y = int(r["picked_left_y"]) if "picked_left_y" in matched_df.columns else int(r["picked_y"]) if "picked_y" in matched_df.columns else int(r["y"])
        plt.gca().add_patch(plt.Circle((x, y), radius=spot_radius, fill=False, edgecolor='cyan', linewidth=1.0))
    plt.title(f"Matched-event spots overlay (n={len(matched_df)})")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


## 7. Reusable analysis pipeline

The same left/primary-channel spot detection, trace extraction and event detection functions are applied in both acquisition modes. In dual-channel mode the secondary pixel is selected within a neighborhood anchored at the corresponding left spot/pixel; secondary-channel spots are **never independently detected**.

In [ ]:
def detect_validated_events(trace, filter_params):
    """Detect candidate positive transitions and optionally validate sustained increases."""
    events = get_events_combined(trace, EVENT_PARAMS, LEVEL_SHIFT_PARAMS)
    if filter_params.get("enabled", False):
        params = {k: v for k, v in filter_params.items() if k != "enabled"}
        events = filter_step_events(trace, events, **params)
    return events


def analyze_recording(input_tiff, output_dir, acquisition_mode="dual", frame_interval_s=None):
    """Analyze a single-channel or horizontally stitched dual-channel TIFF recording.

    Parameters
    ----------
    input_tiff : str or Path
        Input multi-page TIFF sequence.
    output_dir : str or Path
        Directory for exported CSV results.
    acquisition_mode : {"single", "dual"}
        For "dual", every TIFF frame is split at half width.
    frame_interval_s : positive float or None
        Time between consecutive frames in seconds; when None, report frame delays only.

    Returns
    -------
    dict
        DataFrames for spots, all matched pairs and chronological pairs; averaged image.
    """
    if acquisition_mode not in ("single", "dual"):
        raise ValueError("acquisition_mode must be 'single' or 'dual'")
    if frame_interval_s is not None and frame_interval_s <= 0:
        raise ValueError("frame_interval_s must be positive")
    input_tiff = Path(input_tiff).expanduser()
    output_dir = Path(output_dir).expanduser()
    if not input_tiff.is_file():
        raise FileNotFoundError(f"Input TIFF not found: {input_tiff}")
    output_dir.mkdir(parents=True, exist_ok=True)

    volume = read_tiff_stack_gray(str(input_tiff))
    if acquisition_mode == "dual":
        primary, secondary = split_left_right(volume)
    else:
        primary, secondary = volume, None

    primary_average = average_first_n(primary, CHECK_FIRST_N_FRAMES)
    primary_background = estimate_background(primary_average, ROLLING_BALL_RADIUS)
    if secondary is not None:
        secondary_background = estimate_background(
            average_first_n(secondary, CHECK_FIRST_N_FRAMES), ROLLING_BALL_RADIUS
        )

    centers = detect_spots_local_max(
        primary_average, SPOT_RADIUS_PX, NOISE_TOLERANCE, CORNER_K
    )
    print(f"Primary-channel spots detected: {len(centers)}")
    if SHOW_DETECTION_OVERLAY:
        show_detection_compare(primary_average, centers, SPOT_RADIUS_PX)

    derivative_pixel_params = {
        k: EVENT_PARAMS[k] for k in ("t_start", "smooth_N", "spline_k", "dev_smooth_N")
    }
    rows, pair_rows = [], []
    for spot_id, (y0, x0) in enumerate(tqdm(centers, desc="Spots", unit="spot"), start=1):
        row = {"spot_id": spot_id, "x": int(x0), "y": int(y0)}
        if ROI_MODE in ("mean_disk", "p90_disk", "max_disk"):
            left_trace = extract_trace_for_roi(
                primary, primary_background, y0, x0, SPOT_RADIUS_PX, ROI_MODE
            )
            left_y, left_x = y0, x0
            left_score = np.nan
            # Preserve source behavior for non-best-pixel modes: identical fixed ROI in both channels.
            if secondary is not None:
                right_trace = extract_trace_for_roi(
                    secondary, secondary_background, y0, x0, SPOT_RADIUS_PX, ROI_MODE
                )
                right_y, right_x = y0, x0
                right_score = np.nan
        elif ROI_MODE == "best_pixel_by_derivative":
            left_pixels, ys, xs = extract_pixel_traces(
                primary, primary_background, y0, x0, SPOT_RADIUS_PX
            )
            best_idx, left_score, left_y, left_x = choose_best_pixel_by_derivative(
                left_pixels, ys, xs, score_window=BESTPIX_SCORE_WINDOW,
                dev_params=derivative_pixel_params,
                min_peak_intensity=BESTPIX_MIN_PEAK_INTENSITY,
            )
            left_trace = left_pixels[best_idx]
        else:
            raise ValueError(f"Unrecognized ROI_MODE: {ROI_MODE}")

        # Shared event detector for primary-channel hemifusion readout.
        left_events = detect_validated_events(left_trace, LEFT_EVENT_FILTER_PARAMS)
        row.update({
            "picked_left_y": int(left_y), "picked_left_x": int(left_x),
            "picked_y": int(left_y), "picked_x": int(left_x),
            "picked_score": float(left_score), "picked_left_score": float(left_score),
            "n_left_events": len(left_events),
            "left_events": ";".join(map(str, left_events)),
        })

        if secondary is not None:
            if ROI_MODE == "best_pixel_by_derivative":
                if RIGHT_SEARCH_CENTER == "picked_left":
                    anchor_y, anchor_x = left_y, left_x
                elif RIGHT_SEARCH_CENTER == "spot_center":
                    anchor_y, anchor_x = y0, x0
                else:
                    raise ValueError(f"Unrecognized RIGHT_SEARCH_CENTER: {RIGHT_SEARCH_CENTER}")
                right_pixels, ys, xs = extract_pixel_traces(
                    secondary, secondary_background, anchor_y, anchor_x,
                    RIGHT_SEARCH_RADIUS_PX,
                )
                if RIGHT_BESTPIX_INDEPENDENT:
                    if left_events:
                        r_idx, right_score, right_y, right_x = choose_best_pixel_near_events(
                            right_pixels, ys, xs, anchor_events=left_events,
                            half_window=RIGHT_BESTPIX_EVENT_HALF_WINDOW,
                            dev_params=derivative_pixel_params,
                            min_peak_intensity=BESTPIX_MIN_PEAK_INTENSITY,
                        )
                    else:
                        r_idx, right_score, right_y, right_x = choose_best_pixel_by_derivative(
                            right_pixels, ys, xs, score_window=BESTPIX_SCORE_WINDOW,
                            dev_params=derivative_pixel_params,
                            min_peak_intensity=BESTPIX_MIN_PEAK_INTENSITY,
                        )
                    right_trace = right_pixels[r_idx]
                else:
                    right_y, right_x = left_y, left_x
                    right_score = np.nan
                    right_trace = (
                        secondary[:, right_y, right_x] - secondary_background[right_y, right_x]
                    ).astype(float)
                row.update({
                    "right_search_anchor_y": int(anchor_y),
                    "right_search_anchor_x": int(anchor_x),
                    "right_search_radius": int(RIGHT_SEARCH_RADIUS_PX),
                })
            right_events = detect_validated_events(right_trace, RIGHT_EVENT_FILTER_PARAMS)
            matched_pairs = match_events(left_events, right_events, tol=MATCH_TOL)
            row.update({
                "picked_right_y": int(right_y), "picked_right_x": int(right_x),
                "picked_right_score": float(right_score),
                "n_right_events": len(right_events),
                "n_matched_events": len(matched_pairs),
                "right_events": ";".join(map(str, right_events)),
                "matched_events": ";".join(f"{l}|{r}" for l, r in matched_pairs),
            })
            for left_frame, right_frame in matched_pairs:
                delta = right_frame - left_frame
                pair_rows.append({
                    "spot_id": spot_id, "hemifusion_frame": left_frame,
                    "right_event_frame": right_frame, "delta_frames": delta,
                    "right_after_left": delta > 0,
                    "delta_t_s": delta * frame_interval_s if frame_interval_s is not None else np.nan,
                })
            for frame, value in enumerate(right_trace):
                row[f"right_t{frame}"] = float(value)

        for frame, value in enumerate(left_trace):
            row[f"left_t{frame}"] = float(value)
        rows.append(row)

    spots = pd.DataFrame(rows)
    column_names = ["spot_id", "hemifusion_frame", "right_event_frame", "delta_frames", "right_after_left", "delta_t_s"]
    pairs = pd.DataFrame(pair_rows, columns=column_names)
    chronological_pairs = pairs[pairs["right_after_left"]].copy()
    spots.to_csv(output_dir / "spot_traces_and_events.csv", index=False)
    if secondary is not None:
        pairs.to_csv(output_dir / "matched_event_pairs.csv", index=False)
        chronological_pairs.to_csv(output_dir / "chronological_event_pairs.csv", index=False)

    # Save the exact analysis parameters to accompany exported data.
    parameters = {
        "input_tiff": str(input_tiff), "acquisition_mode": acquisition_mode,
        "frame_interval_s": frame_interval_s, "reference_frames": CHECK_FIRST_N_FRAMES,
        "spot_radius_px": SPOT_RADIUS_PX, "noise_tolerance": NOISE_TOLERANCE,
        "rolling_ball_radius": ROLLING_BALL_RADIUS, "corner_k": CORNER_K,
        "roi_mode": ROI_MODE, "best_pixel_score_window": BESTPIX_SCORE_WINDOW,
        "best_pixel_min_peak_intensity": BESTPIX_MIN_PEAK_INTENSITY,
        "right_best_pixel_independent": RIGHT_BESTPIX_INDEPENDENT,
        "right_best_pixel_event_half_window": RIGHT_BESTPIX_EVENT_HALF_WINDOW,
        "right_search_radius_px": RIGHT_SEARCH_RADIUS_PX,
        "right_search_center": RIGHT_SEARCH_CENTER,
        "derivative_detector": EVENT_PARAMS,
        "level_shift_detector": LEVEL_SHIFT_PARAMS,
        "primary_event_filter": LEFT_EVENT_FILTER_PARAMS,
        "secondary_event_filter": RIGHT_EVENT_FILTER_PARAMS,
        "event_matching_tolerance_frames": MATCH_TOL,
        "event_matching_rule": "greedy closest absolute frame difference (no chronology constraint)",
    }
    (output_dir / "analysis_parameters.json").write_text(json.dumps(parameters, indent=2), encoding="utf-8")
    print(f"Saved {len(spots)} spot records and {len(pairs)} matched event pairs to {output_dir}")
    return {"spots": spots, "pairs": pairs, "chronological_pairs": chronological_pairs,
            "average_primary": primary_average}


## 8. Run the analysis

Set the input path and mode in the configuration cell, then run this cell. 

In [ ]:
results = analyze_recording(
    INPUT_TIFF, OUTPUT_DIR,
    acquisition_mode=ACQUISITION_MODE,
    frame_interval_s=FRAME_INTERVAL_S,
)
spots_df = results["spots"]
pairs_df = results["pairs"]
chronological_df = results["chronological_pairs"]
print(f"Spots: {len(spots_df)} | Matched pairs: {len(pairs_df)} | Chronological pairs: {len(chronological_df)}")


## 9. Diagnostic plots and delay histogram

Diagnostic plots independently normalize traces (1st to 99th percentile) **for visualization only**. The saved intensity traces are background-corrected but not display-normalized. Histogram bins are chosen for visualization and are not part of the event detector.

In [ ]:
def plot_spot_events(spots, spot_ids, mode="dual"):
    """Plot one or more spot traces with detected event-frame markers."""
    for _, row in spots[spots["spot_id"].isin(spot_ids)].iterrows():
        lt_cols = sorted((c for c in spots.columns if c.startswith("left_t")),
                         key=lambda s: int(s.split("left_t")[1]))
        lt = row[lt_cols].to_numpy(dtype=float)
        fig, ax = plt.subplots(figsize=(9, 3))
        ax.plot(normalize_trace_for_plot(lt), label="Primary (hemifusion)")
        for e in str(row["left_events"]).split(";"):
            if e:
                ax.axvline(int(e), color="tab:blue", linestyle="--", alpha=0.5)
        if mode == "dual":
            rt_cols = sorted((c for c in spots.columns if c.startswith("right_t")),
                             key=lambda s: int(s.split("right_t")[1]))
            rt = row[rt_cols].to_numpy(dtype=float)
            ax.plot(normalize_trace_for_plot(rt), label="Secondary")
            for e in str(row["right_events"]).split(";"):
                if e:
                    ax.axvline(int(e), color="tab:orange", linestyle=":", alpha=0.5)
        ax.set(xlabel="Frame index", ylabel="Normalized intensity (display only)",
               title=f"Spot {int(row['spot_id'])}")
        ax.legend()
        plt.tight_layout()
        plt.show()


def plot_delay_histogram(chronological_pairs, frame_interval_s=None, bins=30):
    """Visualize chronological matched-event delays; no model fitting is performed."""
    if chronological_pairs.empty:
        print("No chronological matched events available for a histogram.")
        return
    if frame_interval_s is None:
        data = chronological_pairs["delta_frames"].to_numpy()
        xlabel = "Right event − hemifusion (frames)"
    else:
        data = chronological_pairs["delta_t_s"].dropna().to_numpy()
        xlabel = "Right event − hemifusion (s)"
    plt.figure(figsize=(6, 4))
    plt.hist(data, bins=bins)
    plt.xlabel(xlabel)
    plt.ylabel("Event-pair count")
    plt.tight_layout()
    plt.show()
